# 00 · 데이터 준비 (Colab, Drive 없이)

> 학습에 쓸 **두 데이터셋을 `/content` 로 받아** 잘 읽히는지 확인합니다.
> **Google Drive 는 쓰지 않습니다** (사지방 등 Drive 가 없거나 꽉 찬 환경 기준).

받는 데이터:
- **CholecSeg8k** (분할, ~3.1GB) → HuggingFace 캐시 `./data/cholecseg8k`
- **Endoscapes2023** (CVS, ~5.9GB) → `/content/endoscapes2023`

⚠️ Colab 런타임이 완전 초기화되면 `/content` 가 비워져 **다시 받아야** 합니다
(같은 세션 안에선 캐시 재사용). Drive 보존은 맨 아래 메모 참고.

## 0. 환경 준비

Colab 은 매번 빈 컴퓨터이므로 코드를 내려받고 라이브러리를 설치합니다.
여러 번 실행해도 안전합니다 (이미 있으면 최신 코드만 당겨옴).

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
!bash scripts/colab_setup.sh
print("준비 완료 ·", os.getcwd())

## 1. 두 데이터셋 다운로드 (→ /content)

- **CholecSeg8k** 는 HuggingFace 에서 로컬 캐시로 받습니다 (한 번 받으면 재사용).
- **Endoscapes2023** 은 CAMMA 공개 미러에서 zip 으로 받아 `/content` 에 풉니다
  (`wget -c` 로 이어받기, `unzip -o` 로 덮어쓰기).

⚠️ 다운로드(특히 Endoscapes ~5.9GB)가 이 노트북에서 가장 오래 걸립니다.

In [ ]:
# CholecSeg8k (~3.1 GB) -> 로컬 HuggingFace 캐시
!bash scripts/download_cholecseg8k.sh

# Endoscapes2023 (~5.9 GB) -> /content
!wget -q -c -O /content/endoscapes.zip https://s3.unistra.fr/camma_public/datasets/endoscapes/endoscapes.zip
!unzip -q -o /content/endoscapes.zip -d /content/endoscapes2023

## 2. 둘 다 잘 불러와지나 확인

CholecSeg8k 3개 split(합계 8080) + Endoscapes 3개 split 프레임 수가 0이 아니면 성공.

In [ ]:
from src.data.cholecseg8k import CholecSeg8kDataset
from src.data.endoscapes import Endoscapes2023Dataset

for s in ("train", "val", "test"):
    print(f"CholecSeg8k {s:5s}: {len(CholecSeg8kDataset(split=s))} frames")
for s in ("train", "val", "test"):
    ds = Endoscapes2023Dataset(root="/content/endoscapes2023", split=s)
    print(f"Endoscapes  {s:5s}: {len(ds)} CVS keyframes")

## 다음 단계 / 메모

- 데이터가 준비됐으니 공부 노트북 **01~05**, 또는 실행용 **06_smoke_test** /
  **run_pipeline** 로 진행하세요. 모두 같은 경로(HF 캐시, `/content/endoscapes2023`)를 씁니다.
- **Drive 에 보존하고 싶다면**(런타임 초기화에도 유지): `src/utils/colab.py` 의
  `mount_drive()` 로 `data/`·`outputs/` 를 Drive 에 연결할 수 있습니다. 단 Drive
  무료 용량(15GB)을 넘기지 않게 주의 — 본 시리즈는 **Drive 없이 `/content`** 가 기본입니다.